# Notebook 19 - Exercise 4: Automatically Discovered Non-Parametric Predictive Structure

Notebook 19 repeats Linus's three empirical steps for Random Forest, Gradient Boosting, MLP, and Transformer models.

1. **Discover** each model's nonlinear predictive parent blocks and lag order on TRAIN → VALIDATION common support.
2. **Freeze and forecast** with independently selected hyperparameters, then evaluate TEST once.
3. **Refit eventless backgrounds** with the discovered specification unchanged and estimate event weights using corrected Notebook 18 timing.

“Structure” means conditional predictive or Granger-style structure: whether a lagged source improves out-of-sample prediction conditional on retained histories. It is not structural economic causality. Transformer attention masking prevents use of future tokens; attention weights are not treated as causal edges.

## 19.0 Scope, conventions, and leakage contract

The maximum lag of 12 is a search ceiling, not a supplied final order. All lags are constructed on the intact chronology before any split filter or event mask. TEST is excluded from every selection decision, N16 results are loaded only after structures freeze, and N18 event masks enter only in Step 3.

In [1]:
from pathlib import Path
from copy import deepcopy
import json
import math
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 120)
warnings.filterwarnings("ignore", message="enable_nested_tensor")
torch.set_num_threads(1)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "Data" / "processed").exists())
PROCESSED = ROOT / "Data" / "processed"
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
RUN_ID = pd.Timestamp.now(tz="UTC").isoformat()

L_MAX = 12
STATES = ("Q", "R", "I")
TARGETS = ("R", "I")
MODEL_FAMILIES = ("RF", "GB", "MLP", "TRANSFORMER")
FAMILIES = ("FOMC", "US Employment", "US CPI", "BoJ", "JPY National CPI", "JPY GDP")
SEEDS = (19, 119, 219)
MAX_EPOCHS = 400
PATIENCE = 40
MIN_DELTA = 1e-5
BATCH_SIZE = 64
PARSIMONY_BAND = 1.01

SPLIT_BOUNDS = {
    "train": (pd.Timestamp("2003-05-05"), pd.Timestamp("2017-03-27")),
    "validation": (pd.Timestamp("2017-03-28"), pd.Timestamp("2021-11-12")),
    "test": (pd.Timestamp("2021-11-15"), pd.Timestamp("2026-07-02")),
}

panel = pd.read_csv(PROCESSED / "16_empirical_analysis_panel.csv", parse_dates=["model_day"])
panel = panel.sort_values("model_day").reset_index(drop=True)
panel["Q"] = pd.to_numeric(panel["squared_return"])
panel["R"] = pd.to_numeric(panel["realised_variance_ann_252"])
panel["I"] = pd.to_numeric(panel["iv_model_var"])
assert panel["model_day"].is_unique and panel["model_day"].is_monotonic_increasing
for split, (start, end) in SPLIT_BOUNDS.items():
    observed = panel.loc[panel["sample_split"].eq(split), "model_day"]
    assert observed.min() == start and observed.max() == end

selection_audit = {
    "structure_selection_split": "TRAIN_TO_VALIDATION",
    "hyperparameter_selection_split": "TRAIN_TO_VALIDATION",
    "test_used_for_selection": False,
    "n16_structure_used_for_selection": False,
    "event_mask_used_for_structure_selection": False,
    "event_mask_used_for_hyperparameter_selection": False,
    "l_max": L_MAX,
    "lag_order_is_data_selected": True,
}
scope_audit = pd.DataFrame([
    ("Targets", "R, I"),
    ("Models", ", ".join(MODEL_FAMILIES)),
    ("TRAIN / VALIDATION / TEST N", " / ".join(str(int(panel["sample_split"].eq(s).sum())) for s in ["train", "validation", "test"])),
    ("Maximum candidate lag", "12 (search ceiling)"),
    ("R parent candidates", "{R}, {R,Q}, {R,I}, {R,Q,I}"),
    ("I parent candidates", "{I}, {I,Q}, {I,R}, {I,Q,R}"),
    ("Eventless timing source", "Corrected N18 same-model-day mapping, loaded only in Step 3"),
    ("Leakage rule", "TRAIN fits; VALIDATION selects; TEST evaluated once"),
], columns=["Audit item", "Frozen convention"])
display(scope_audit)
print("Execution device:", DEVICE)

,Audit item,Frozen convention
0,Targets,"R, I"
1,Models,"RF, GB, MLP, TRANSFORMER"
2,TRAIN / VALIDATION / TEST N,3626 / 1209 / 1209
3,Maximum candidate lag,12 (search ceiling)
4,R parent candidates,"{R}, {R,Q}, {R,I}, {R,Q,I}"
5,I parent candidates,"{I}, {I,Q}, {I,R}, {I,Q,R}"
6,Eventless timing source,"Corrected N18 same-model-day mapping, loaded o..."
7,Leakage rule,TRAIN fits; VALIDATION selects; TEST evaluated...


Execution device: cuda


## 19.1 Common data and structure-search support

The full Q, R, and I lag panel is built first. Candidate structures for a given target are then compared on identical TRAIN and VALIDATION rows requiring the target and every lag 1–12 of all three states to be finite. No event mask is present in this stage.

In [2]:
state_snapshot = panel[["model_day", "Q", "R", "I"]].copy()
MAX_LAG_FEATURES = [f"{state}_lag{lag}" for state in STATES for lag in range(1, L_MAX + 1)]
for state in STATES:
    for lag in range(1, L_MAX + 1):
        panel[f"{state}_lag{lag}"] = panel[state].shift(lag)
lag_snapshot = panel[["model_day", *MAX_LAG_FEATURES]].copy()
assert panel[["model_day", "Q", "R", "I"]].equals(state_snapshot)
assert L_MAX == 12

PARENT_SETS = {
    "R": (("R",), ("R", "Q"), ("R", "I"), ("R", "Q", "I")),
    "I": (("I",), ("I", "Q"), ("I", "R"), ("I", "Q", "R")),
}
SEARCH_SUPPORT = {}
support_rows = []
for target in TARGETS:
    finite = np.isfinite(panel[[target, *MAX_LAG_FEATURES]].to_numpy(dtype=float)).all(axis=1)
    for split in ("train", "validation"):
        frame = panel.loc[finite & panel["sample_split"].eq(split)].copy()
        SEARCH_SUPPORT[(target, split)] = frame
        support_rows.append({
            "target": target,
            "split": split,
            "raw_split_n": int(panel["sample_split"].eq(split).sum()),
            "common_structure_support_n": len(frame),
            "finite_target_required": True,
            "finite_all_QRI_lags_1_12_required": True,
            "event_mask_applied": False,
            "lags_built_before_split_filter": True,
            "selection_support_evaluated": True,
        })
    support_rows.append({
        "target": target,
        "split": "test",
        "raw_split_n": int(panel["sample_split"].eq("test").sum()),
        "common_structure_support_n": np.nan,
        "finite_target_required": False,
        "finite_all_QRI_lags_1_12_required": False,
        "event_mask_applied": False,
        "lags_built_before_split_filter": True,
        "selection_support_evaluated": False,
    })
input_and_support_audit = pd.DataFrame(support_rows)
assert not any(column.startswith(("R_is_", "I_is_")) for column in panel.columns)
assert not input_and_support_audit.loc[input_and_support_audit["split"].eq("test"), "selection_support_evaluated"].any()
display(input_and_support_audit)

,target,split,raw_split_n,common_structure_support_n,finite_target_required,finite_all_QRI_lags_1_12_required,event_mask_applied,lags_built_before_split_filter,selection_support_evaluated
0,R,train,3626,2200.0,True,True,False,True,True
1,R,validation,1209,1078.0,True,True,False,True,True
2,R,test,1209,NaN,False,False,False,True,False
3,I,train,3626,2226.0,True,True,False,True,True
4,I,validation,1209,1087.0,True,True,False,True,True
5,I,test,1209,NaN,False,False,False,True,False


## 19.2 Step 1 — automatic nonlinear predictive-structure discovery

For each model and target, 48 candidates are evaluated: four parent-block sets crossed with lag orders 1–12. RF and GB use the first grid configuration as a deterministic probe. MLP and Transformer use their first configuration under all three frozen seeds; mean VALIDATION RMSE selects structure. TEST, N16 structure results, and event masks are unavailable to this code.

The pre-frozen parsimony rule selects the fewest cross-series blocks and then the shortest lag inside 1% of the minimum VALIDATION RMSE.

In [3]:
RF_GRID = {
    "RF_01": {"max_depth": 8, "min_samples_leaf": 5},
    "RF_02": {"max_depth": 8, "min_samples_leaf": 10},
    "RF_03": {"max_depth": None, "min_samples_leaf": 5},
    "RF_04": {"max_depth": None, "min_samples_leaf": 10},
}
GB_GRID = {
    "GB_01": {"n_estimators": 200, "learning_rate": 0.03},
    "GB_02": {"n_estimators": 200, "learning_rate": 0.05},
    "GB_03": {"n_estimators": 400, "learning_rate": 0.03},
    "GB_04": {"n_estimators": 400, "learning_rate": 0.05},
}
MLP_GRID = {
    "MLP_01": ([32], 1e-3),
    "MLP_02": ([32], 3e-4),
    "MLP_03": ([32, 16], 1e-3),
    "MLP_04": ([32, 16], 3e-4),
}
TR_GRID = {
    "TR_01": ((16, 2, 32), 1e-3),
    "TR_02": ((16, 2, 32), 3e-4),
    "TR_03": ((24, 4, 48), 1e-3),
    "TR_04": ((24, 4, 48), 3e-4),
}
GRIDS = {"RF": RF_GRID, "GB": GB_GRID, "MLP": MLP_GRID, "TRANSFORMER": TR_GRID}
PROBE_CONFIG = {model: next(iter(GRIDS[model])) for model in MODEL_FAMILIES}
RF_FIXED = {
    "n_estimators": 500, "max_features": 0.5, "criterion": "squared_error",
    "bootstrap": True, "random_state": 19, "n_jobs": -1,
}
GB_FIXED = {
    "max_depth": 2, "min_samples_leaf": 10, "subsample": 1.0,
    "loss": "squared_error", "random_state": 19,
}
probe_configuration_table = pd.DataFrame([
    {"model": model, "probe_candidate_id": config_id, "probe_hyperparameters": str(GRIDS[model][config_id])}
    for model, config_id in PROBE_CONFIG.items()
])
display(probe_configuration_table)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def feature_columns(parents, lag_order):
    return [f"{state}_lag{lag}" for lag in range(lag_order, 0, -1) for state in parents]

def model_frame(target, split, parents, lag_order, common_support=False):
    if common_support:
        return SEARCH_SUPPORT[(target, split)].copy()
    columns = [target, *feature_columns(parents, lag_order)]
    finite = np.isfinite(panel[columns].to_numpy(dtype=float)).all(axis=1)
    return panel.loc[finite & panel["sample_split"].eq(split)].copy()

def compute_metrics(actual, prediction, persistence=None):
    actual = np.asarray(actual, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    valid = np.isfinite(actual) & np.isfinite(prediction)
    actual = actual[valid]
    prediction = prediction[valid]
    error = actual - prediction
    positive = (actual > 0) & (prediction > 0)
    qlike = np.nan
    if positive.any():
        ratio = actual[positive] / prediction[positive]
        qlike = float(np.mean(ratio - np.log(ratio) - 1))
    oos_r2 = np.nan
    if persistence is not None:
        persistence = np.asarray(persistence, dtype=float)[valid]
        paired = np.isfinite(persistence)
        denominator = np.sum((actual[paired] - persistence[paired]) ** 2)
        if denominator > 0:
            oos_r2 = float(1 - np.sum((actual[paired] - prediction[paired]) ** 2) / denominator)
    return {
        "valid_forecast_n": int(len(actual)),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "MAE": float(np.mean(np.abs(error))),
        "QLIKE": qlike,
        "QLIKE_valid_n": int(positive.sum()),
        "OOS_R2": oos_r2,
        "nonpositive_forecast_n": int((prediction <= 0).sum()),
    }

def make_tree(model, config_id):
    if model == "RF":
        return RandomForestRegressor(**RF_FIXED, **RF_GRID[config_id])
    return GradientBoostingRegressor(**GB_FIXED, **GB_GRID[config_id])

def fit_scaler(frame, target, parents, lag_order):
    scale = {}
    for state in parents:
        columns = [f"{state}_lag{lag}" for lag in range(1, lag_order + 1)]
        values = frame[columns].to_numpy(dtype=float).ravel()
        std = float(values.std())
        assert std > 0
        scale[state] = (float(values.mean()), std)
    target_values = frame[target].to_numpy(dtype=float)
    target_std = float(target_values.std())
    assert target_std > 0
    scale["TARGET"] = (float(target_values.mean()), target_std)
    return scale

def scaled_features(frame, columns, scale):
    values = frame[columns].to_numpy(dtype=float).copy()
    for index, column in enumerate(columns):
        state = column.split("_lag")[0]
        mean, std = scale[state]
        values[:, index] = (values[:, index] - mean) / std
    return values

def scaled_feature_array(values, columns, scale):
    values = np.asarray(values, dtype=float).copy()
    for index, column in enumerate(columns):
        state = column.split("_lag")[0]
        mean, std = scale[state]
        values[:, index] = (values[:, index] - mean) / std
    return values

def scale_target(values, scale):
    mean, std = scale["TARGET"]
    return (np.asarray(values, dtype=float) - mean) / std

def unscale_target(values, scale):
    mean, std = scale["TARGET"]
    return np.asarray(values, dtype=float) * std + mean

class MLPRegressor(nn.Module):
    def __init__(self, input_width, hidden_layers):
        super().__init__()
        layers = []
        width = input_width
        for hidden in hidden_layers:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(0.1)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.network(inputs).squeeze(-1)

class TransformerRegressor(nn.Module):
    def __init__(self, input_dimension, sequence_length, d_model, n_heads, ff_dim):
        super().__init__()
        self.embedding = nn.Linear(input_dimension, d_model)
        positions = torch.arange(sequence_length).unsqueeze(1)
        divisors = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        encoding = torch.zeros(sequence_length, d_model)
        encoding[:, 0::2] = torch.sin(positions * divisors)
        encoding[:, 1::2] = torch.cos(positions * divisors)
        self.register_buffer("positional_encoding", encoding.unsqueeze(0))
        no_future_mask = torch.triu(torch.ones(sequence_length, sequence_length, dtype=torch.bool), diagonal=1)
        self.register_buffer("no_future_mask", no_future_mask)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=0.1, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.head = nn.Linear(d_model, 1)

    def forward(self, inputs):
        hidden = self.encoder(self.embedding(inputs) + self.positional_encoding, mask=self.no_future_mask)
        return self.head(hidden[:, -1]).squeeze(-1)

def make_neural(model, config_id, input_width, parent_count, lag_order):
    configuration = GRIDS[model][config_id]
    architecture, learning_rate = configuration
    if model == "MLP":
        network = MLPRegressor(input_width, architecture)
    else:
        network = TransformerRegressor(parent_count, lag_order, *architecture)
    return network.to(DEVICE), learning_rate

def neural_tensor(values, model, lag_order, parent_count):
    tensor = torch.tensor(values, dtype=torch.float32)
    if model == "TRANSFORMER":
        tensor = tensor.reshape(-1, lag_order, parent_count)
    return tensor

def train_neural_early(model, config_id, seed, train_frame, validation_frame, target, parents, lag_order):
    set_seed(seed)
    columns = feature_columns(parents, lag_order)
    scale = fit_scaler(train_frame, target, parents, lag_order)
    train_x = neural_tensor(scaled_features(train_frame, columns, scale), model, lag_order, len(parents))
    train_y = torch.tensor(scale_target(train_frame[target], scale), dtype=torch.float32)
    validation_x = neural_tensor(scaled_features(validation_frame, columns, scale), model, lag_order, len(parents)).to(DEVICE)
    validation_y = torch.tensor(scale_target(validation_frame[target], scale), dtype=torch.float32).to(DEVICE)
    network, learning_rate = make_neural(model, config_id, len(columns), len(parents), lag_order)
    optimizer = torch.optim.AdamW(network.parameters(), lr=learning_rate, weight_decay=1e-4)
    loss_function = nn.MSELoss()
    loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_state, best_epoch, wait = float("inf"), None, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        network.train()
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_function(network(batch_x), batch_y)
            loss.backward()
            if model == "TRANSFORMER":
                torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
            optimizer.step()
        network.eval()
        with torch.no_grad():
            validation_loss = loss_function(network(validation_x), validation_y).item()
        if validation_loss < best_loss - MIN_DELTA:
            best_loss = validation_loss
            best_state = {name: value.detach().clone() for name, value in network.state_dict().items()}
            best_epoch, wait = epoch, 0
        else:
            wait += 1
        if wait >= PATIENCE:
            break
    assert best_state is not None
    network.load_state_dict(best_state)
    return network, scale, best_epoch

def predict_neural(network, frame_or_array, columns, scale, model, lag_order, parent_count):
    if isinstance(frame_or_array, pd.DataFrame):
        values = scaled_features(frame_or_array, columns, scale)
    else:
        values = scaled_feature_array(frame_or_array, columns, scale)
    inputs = neural_tensor(values, model, lag_order, parent_count).to(DEVICE)
    network.eval()
    with torch.no_grad():
        prediction = network(inputs).detach().cpu().numpy()
    return unscale_target(prediction, scale)

def train_neural_fixed(model, config_id, seed, calibration, target, parents, lag_order, epochs):
    set_seed(seed)
    columns = feature_columns(parents, lag_order)
    scale = fit_scaler(calibration, target, parents, lag_order)
    train_x = neural_tensor(scaled_features(calibration, columns, scale), model, lag_order, len(parents))
    train_y = torch.tensor(scale_target(calibration[target], scale), dtype=torch.float32)
    network, learning_rate = make_neural(model, config_id, len(columns), len(parents), lag_order)
    optimizer = torch.optim.AdamW(network.parameters(), lr=learning_rate, weight_decay=1e-4)
    loss_function = nn.MSELoss()
    loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
    network.train()
    for _ in range(int(epochs)):
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_function(network(batch_x), batch_y)
            loss.backward()
            if model == "TRANSFORMER":
                torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
            optimizer.step()
    return network, scale

def ensemble_prediction(models, frame_or_array, columns, scales, model, lag_order, parent_count):
    predictions = [
        predict_neural(network, frame_or_array, columns, scale, model, lag_order, parent_count)
        for network, scale in zip(models, scales)
    ]
    return np.mean(np.vstack(predictions), axis=0)

def save_figure(fig, filename):
    assert filename.startswith("19_figure_") and filename.endswith(".png")
    FIGURES.mkdir(parents=True, exist_ok=True)
    path = FIGURES / filename
    if path.exists():
        assert path.is_file()
        path.unlink()
    fig.savefig(str(path), dpi=160, bbox_inches="tight")
    plt.close(fig)
    return path

,model,probe_candidate_id,probe_hyperparameters
0,RF,RF_01,"{'max_depth': 8, 'min_samples_leaf': 5}"
1,GB,GB_01,"{'n_estimators': 200, 'learning_rate': 0.03}"
2,MLP,MLP_01,"([32], 0.001)"
3,TRANSFORMER,TR_01,"((16, 2, 32), 0.001)"


In [4]:
structure_rows = []
neural_seed_records = []
for model in MODEL_FAMILIES:
    config_id = PROBE_CONFIG[model]
    for target in TARGETS:
        train_frame = SEARCH_SUPPORT[(target, "train")]
        validation_frame = SEARCH_SUPPORT[(target, "validation")]
        for parents in PARENT_SETS[target]:
            for lag_order in range(1, L_MAX + 1):
                columns = feature_columns(parents, lag_order)
                candidate_id = f"{target}__{'+'.join(parents)}__L{lag_order:02d}"
                if model in {"RF", "GB"}:
                    estimator = make_tree(model, config_id)
                    estimator.fit(train_frame[columns], train_frame[target])
                    prediction = estimator.predict(validation_frame[columns])
                    metric = compute_metrics(validation_frame[target], prediction)
                    seed_count = 1
                else:
                    seed_metrics = []
                    for seed in SEEDS:
                        network, scale, best_epoch = train_neural_early(
                            model, config_id, seed, train_frame, validation_frame,
                            target, parents, lag_order,
                        )
                        prediction = predict_neural(network, validation_frame, columns, scale, model, lag_order, len(parents))
                        metric_seed = compute_metrics(validation_frame[target], prediction)
                        neural_seed_records.append({
                            "stage": "STRUCTURE_SEARCH", "model": model, "target": target,
                            "candidate_id": candidate_id, "config_id": config_id, "seed": seed,
                            "parent_blocks": "|".join(parents), "lag_order": lag_order,
                            "best_epoch": best_epoch, "validation_RMSE": metric_seed["RMSE"],
                            "validation_MAE": metric_seed["MAE"],
                        })
                        seed_metrics.append(metric_seed)
                        del network
                    metric = {
                        "RMSE": float(np.mean([row["RMSE"] for row in seed_metrics])),
                        "MAE": float(np.mean([row["MAE"] for row in seed_metrics])),
                    }
                    seed_count = len(SEEDS)
                structure_rows.append({
                    "model": model, "target": target, "candidate_id": candidate_id,
                    "parent_blocks": "|".join(parents),
                    "parent_label": "{" + ",".join(parents) + "}",
                    "cross_parent_count": len(parents) - 1,
                    "lag_order": lag_order, "probe_config_id": config_id,
                    "validation_RMSE": metric["RMSE"], "validation_MAE": metric["MAE"],
                    "seed_count": seed_count,
                })
        print(f"Structure search complete: {model} target {target}")

structure_search_validation = pd.DataFrame(structure_rows)
assert len(structure_search_validation) == 384
assert not structure_search_validation.duplicated(["model", "target", "candidate_id"]).any()
assert structure_search_validation["lag_order"].between(1, L_MAX).all()
assert selection_audit["test_used_for_selection"] is False
assert selection_audit["n16_structure_used_for_selection"] is False
assert selection_audit["event_mask_used_for_structure_selection"] is False

Structure search complete: RF target R


Structure search complete: RF target I


Structure search complete: GB target R


Structure search complete: GB target I


Structure search complete: MLP target R


Structure search complete: MLP target I


Structure search complete: TRANSFORMER target R


Structure search complete: TRANSFORMER target I


In [5]:
structure_search_validation["within_1pct"] = False
structure_search_validation["raw_rmse_winner"] = False
structure_search_validation["selected"] = False
selected_rows = []
for (model, target), group in structure_search_validation.groupby(["model", "target"], sort=False):
    minimum = float(group["validation_RMSE"].min())
    raw_index = group.sort_values(["validation_RMSE", "candidate_id"]).index[0]
    eligible = group.loc[group["validation_RMSE"].le(PARSIMONY_BAND * minimum)]
    selected_index = eligible.sort_values(
        ["cross_parent_count", "lag_order", "validation_RMSE", "candidate_id"]
    ).index[0]
    structure_search_validation.loc[group.index, "within_1pct"] = group["validation_RMSE"].le(PARSIMONY_BAND * minimum)
    structure_search_validation.loc[raw_index, "raw_rmse_winner"] = True
    structure_search_validation.loc[selected_index, "selected"] = True
    row = structure_search_validation.loc[selected_index]
    selected_rows.append({
        "model": model, "target": target, "selected_parent_blocks": row["parent_blocks"],
        "selected_parent_label": row["parent_label"],
        "cross_parent_count": int(row["cross_parent_count"]),
        "selected_lag_order": int(row["lag_order"]),
        "minimum_validation_RMSE": minimum,
        "selected_validation_RMSE": float(row["validation_RMSE"]),
        "selected_to_minimum_ratio": float(row["validation_RMSE"] / minimum),
        "parsimony_changed_raw_winner": bool(selected_index != raw_index),
        "selected_candidate_id": row["candidate_id"],
        "probe_config_id": row["probe_config_id"],
    })
discovered_structures = pd.DataFrame(selected_rows).sort_values(["target", "model"]).reset_index(drop=True)
assert len(discovered_structures) == 8
assert discovered_structures["selected_lag_order"].between(1, 12).all()
for row in discovered_structures.itertuples(index=False):
    assert row.target in row.selected_parent_blocks.split("|")
STRUCTURES_FROZEN = True

for target, filename, title in [
    ("R", "19_figure_19_1_structure_search_R.png", "Figure 19.1. Realised-variance structure search"),
    ("I", "19_figure_19_2_structure_search_I.png", "Figure 19.2. Implied-variance structure search"),
]:
    fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
    image = None
    values = structure_search_validation.loc[structure_search_validation["target"].eq(target), "validation_RMSE"]
    vmin, vmax = float(values.min()), float(values.max())
    for ax, model in zip(axes.ravel(), MODEL_FAMILIES):
        subset = structure_search_validation.loc[
            structure_search_validation["target"].eq(target) & structure_search_validation["model"].eq(model)
        ]
        parent_order = ["{" + ",".join(parents) + "}" for parents in PARENT_SETS[target]]
        matrix = subset.pivot(index="parent_label", columns="lag_order", values="validation_RMSE").reindex(index=parent_order, columns=range(1, 13))
        image = ax.imshow(matrix.to_numpy(), aspect="auto", cmap="viridis_r", vmin=vmin, vmax=vmax)
        chosen = subset.loc[subset["selected"]].iloc[0]
        ax.scatter(int(chosen["lag_order"]) - 1, parent_order.index(chosen["parent_label"]), marker="*", s=220, color="red", edgecolor="white")
        ax.set_xticks(range(12), range(1, 13))
        ax.set_yticks(range(4), parent_order)
        ax.set_xlabel("Lag order L")
        ax.set_title(model)
    fig.colorbar(image, ax=axes, shrink=0.8, label="VALIDATION RMSE")
    fig.suptitle(title)
    save_figure(fig, filename)

print("Table 19.2 — discovered structures")
display(discovered_structures[[
    "model", "target", "selected_parent_label", "selected_lag_order",
    "minimum_validation_RMSE", "selected_validation_RMSE", "parsimony_changed_raw_winner",
]])

Table 19.2 — discovered structures


,model,target,selected_parent_label,selected_lag_order,minimum_validation_RMSE,selected_validation_RMSE,parsimony_changed_raw_winner
0,GB,I,{I},2,0.005172,0.005172,False
1,MLP,I,"{I,Q}",4,0.004388,0.004388,False
2,RF,I,{I},2,0.004449,0.004490,True
3,TRANSFORMER,I,"{I,Q,R}",5,0.004290,0.004290,False
4,GB,R,"{R,I}",2,0.005086,0.005134,True
5,MLP,R,"{R,I}",1,0.004942,0.004942,False
6,RF,R,"{R,I}",1,0.005085,0.005089,True
7,TRANSFORMER,R,"{R,I}",1,0.004940,0.004978,True


The selected table above is the frozen output of the non-parametric search. RF selected {R,I} at L=1 and {I} at L=2; GB selected {R,I} at L=2 and {I} at L=2; MLP selected {R,I} at L=1 and {I,Q} at L=4; Transformer selected {R,I} at L=1 and {I,Q,R} at L=5 for R and I respectively. Parsimony changed the raw minimum-RMSE winner for RF on both targets, GB for R, and Transformer for R. Only now may the classical N16 benchmark be opened.

## 19.3 Post-freeze comparison with N16 GC and TDMI

N16 remains an external benchmark rather than an input. For R, every flexible model agrees with N16 by retaining I and dropping Q; RF, MLP, and Transformer also match N16's lag 1, while GB selects lag 2. For I, the Transformer independently retains both Q and R, fully matching the N16 edge pattern, while MLP retains Q only and RF/GB retain own history only. All flexible I orders (2, 2, 4, and 5) are shorter than N16's order 12. Agreement is corroborating predictive evidence; disagreement may reflect nonlinear relationships or model instability. Neither establishes structural economic causation.

In [6]:
assert STRUCTURES_FROZEN
spec_16 = pd.read_csv(PROCESSED / "16_primary_specification.csv").iloc[0]
tdmi_16 = pd.read_csv(PROCESSED / "16_tdmi_summary.csv")
tdmi_16 = tdmi_16.loc[tdmi_16["primary_specification"].eq(True)].copy()
assert len(tdmi_16) == 4

def as_bool(value):
    return str(value).strip().lower() in {"true", "1"}

edge_contract = [
    ("R", "Q", "retain_Q_to_R", "squared_return", "realised_variance_ann_252"),
    ("R", "I", "retain_I_to_R", "implied_variance_ann", "realised_variance_ann_252"),
    ("I", "Q", "retain_Q_to_I", "squared_return", "implied_variance_ann"),
    ("I", "R", "retain_R_to_I", "realised_variance_ann_252", "implied_variance_ann"),
]
comparison_rows = []
for target, source, field, tdmi_source, tdmi_target in edge_contract:
    tdmi_row = tdmi_16.loc[
        tdmi_16["source"].eq(tdmi_source) & tdmi_16["target"].eq(tdmi_target)
    ].iloc[0]
    record = {
        "row_type": "EDGE", "target": target, "candidate_source": source,
        "N16_GC_retain": as_bool(spec_16[field]),
        "N16_TDMI_summary": (
            f"support={bool(tdmi_row['training_tdmi_support'])}; "
            f"peak_lag={int(tdmi_row['training_tdmi_peak_lag'])}; "
            f"train_value={tdmi_row['training_tdmi_value']:.6g}; "
            f"validation_p={tdmi_row['tdmi_validation_p']:.6g}"
        ),
        "N16_selected_lag_order": np.nan,
    }
    for model in MODEL_FAMILIES:
        parents = discovered_structures.loc[
            discovered_structures["model"].eq(model) & discovered_structures["target"].eq(target),
            "selected_parent_blocks",
        ].iloc[0].split("|")
        record[f"{model}_retain"] = source in parents
        record[f"{model}_selected_lag_order"] = np.nan
    comparison_rows.append(record)

for target, n16_order in [("R", int(spec_16["selected_realised_order"])), ("I", int(spec_16["selected_implied_order"]))]:
    record = {
        "row_type": "LAG_ORDER", "target": target, "candidate_source": "OWN_HISTORY_ORDER",
        "N16_GC_retain": np.nan, "N16_TDMI_summary": "",
        "N16_selected_lag_order": n16_order,
    }
    for model in MODEL_FAMILIES:
        record[f"{model}_retain"] = np.nan
        record[f"{model}_selected_lag_order"] = int(discovered_structures.loc[
            discovered_structures["model"].eq(model) & discovered_structures["target"].eq(target),
            "selected_lag_order",
        ].iloc[0])
    comparison_rows.append(record)
structure_vs_n16 = pd.DataFrame(comparison_rows)
selection_audit["n16_structure_used_for_selection"] = False
assert selection_audit["n16_structure_used_for_selection"] is False

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True, constrained_layout=True)
for ax, target in zip(axes, TARGETS):
    row = structure_vs_n16.loc[
        structure_vs_n16["row_type"].eq("LAG_ORDER") & structure_vs_n16["target"].eq(target)
    ].iloc[0]
    labels = ["N16", *MODEL_FAMILIES]
    values = [row["N16_selected_lag_order"], *[row[f"{model}_selected_lag_order"] for model in MODEL_FAMILIES]]
    bars = ax.bar(labels, values, color=["#555555", "#4472C4", "#70AD47", "#ED7D31", "#A64D79"])
    ax.bar_label(bars, fmt="%.0f")
    ax.set_ylim(0, 13)
    ax.set_title(f"Target {target}")
    ax.set_ylabel("Selected lag order")
    ax.tick_params(axis="x", rotation=25)
fig.suptitle("Figure 19.3. Independently selected lag orders versus N16")
save_figure(fig, "19_figure_19_3_selected_lag_orders.png")
display(structure_vs_n16)

,row_type,target,candidate_source,N16_GC_retain,N16_TDMI_summary,N16_selected_lag_order,RF_retain,RF_selected_lag_order,GB_retain,GB_selected_lag_order,MLP_retain,MLP_selected_lag_order,TRANSFORMER_retain,TRANSFORMER_selected_lag_order
0,EDGE,R,Q,False,support=True; peak_lag=1; train_value=0.062370...,NaN,False,NaN,False,NaN,False,NaN,False,NaN
1,EDGE,R,I,True,support=True; peak_lag=1; train_value=0.308911...,NaN,True,NaN,True,NaN,True,NaN,True,NaN
2,EDGE,I,Q,True,support=True; peak_lag=1; train_value=0.086976...,NaN,False,NaN,False,NaN,True,NaN,True,NaN
3,EDGE,I,R,True,support=True; peak_lag=1; train_value=0.279067...,NaN,False,NaN,False,NaN,False,NaN,True,NaN
4,LAG_ORDER,R,OWN_HISTORY_ORDER,NaN,,1.0,NaN,1.0,NaN,2.0,NaN,1.0,NaN,1.0
5,LAG_ORDER,I,OWN_HISTORY_ORDER,NaN,,12.0,NaN,2.0,NaN,2.0,NaN,4.0,NaN,5.0


## 19.4 Hyperparameter selection after structure freeze

Each discovered parent set and lag order now remains fixed. The original small candidate grid is evaluated with TRAIN fitting and ordinary VALIDATION scoring. Neural candidates retain the three-seed discipline; their final refit duration is the median best epoch across the selected candidate's development seeds. A subsequent diagnostic rechecks all four parent sets at the selected lag under final hyperparameters without changing the frozen structure.

In [7]:
def selected_structure(model, target):
    row = discovered_structures.loc[
        discovered_structures["model"].eq(model) & discovered_structures["target"].eq(target)
    ].iloc[0]
    return tuple(row["selected_parent_blocks"].split("|")), int(row["selected_lag_order"])

hyperparameter_rows = []
selected_hyperparameters = {}
frozen_epochs = {}
for model in MODEL_FAMILIES:
    for target in TARGETS:
        parents, lag_order = selected_structure(model, target)
        columns = feature_columns(parents, lag_order)
        train_frame = model_frame(target, "train", parents, lag_order)
        validation_frame = model_frame(target, "validation", parents, lag_order)
        candidate_results = []
        for config_id in GRIDS[model]:
            if model in {"RF", "GB"}:
                estimator = make_tree(model, config_id)
                estimator.fit(train_frame[columns], train_frame[target])
                prediction = estimator.predict(validation_frame[columns])
                metric = compute_metrics(validation_frame[target], prediction)
                mean_rmse, mean_mae, median_epoch = metric["RMSE"], metric["MAE"], np.nan
                seed_count = 1
            else:
                seed_metrics, seed_epochs = [], []
                for seed in SEEDS:
                    network, scale, best_epoch = train_neural_early(
                        model, config_id, seed, train_frame, validation_frame,
                        target, parents, lag_order,
                    )
                    prediction = predict_neural(network, validation_frame, columns, scale, model, lag_order, len(parents))
                    metric_seed = compute_metrics(validation_frame[target], prediction)
                    neural_seed_records.append({
                        "stage": "HYPERPARAMETER_SELECTION", "model": model, "target": target,
                        "candidate_id": config_id, "config_id": config_id, "seed": seed,
                        "parent_blocks": "|".join(parents), "lag_order": lag_order,
                        "best_epoch": best_epoch, "validation_RMSE": metric_seed["RMSE"],
                        "validation_MAE": metric_seed["MAE"],
                    })
                    seed_metrics.append(metric_seed)
                    seed_epochs.append(best_epoch)
                    del network
                mean_rmse = float(np.mean([row["RMSE"] for row in seed_metrics]))
                mean_mae = float(np.mean([row["MAE"] for row in seed_metrics]))
                median_epoch = int(np.median(seed_epochs))
                seed_count = len(SEEDS)
            result = {
                "model": model, "target": target, "candidate_id": config_id,
                "hyperparameters": str(GRIDS[model][config_id]),
                "parent_blocks": "|".join(parents), "lag_order": lag_order,
                "mean_validation_RMSE": mean_rmse, "mean_validation_MAE": mean_mae,
                "seed_count": seed_count, "median_best_epoch": median_epoch,
                "selected": False,
            }
            hyperparameter_rows.append(result)
            candidate_results.append(result)
        winner = sorted(candidate_results, key=lambda row: (row["mean_validation_RMSE"], row["candidate_id"]))[0]
        winner["selected"] = True
        selected_hyperparameters[(model, target)] = winner["candidate_id"]
        if model in {"MLP", "TRANSFORMER"}:
            frozen_epochs[(model, target)] = int(winner["median_best_epoch"])
        print(f"Hyperparameter selection complete: {model} target {target}")

hyperparameter_selection_validation = pd.DataFrame(hyperparameter_rows)
selected_spec_rows = []
for model in MODEL_FAMILIES:
    for target in TARGETS:
        parents, lag_order = selected_structure(model, target)
        config_id = selected_hyperparameters[(model, target)]
        structure_row = discovered_structures.loc[
            discovered_structures["model"].eq(model) & discovered_structures["target"].eq(target)
        ].iloc[0]
        selected_spec_rows.append({
            "model": model, "target": target,
            "selected_parent_blocks": "|".join(parents),
            "selected_lag_order": lag_order,
            "structure_candidate_id": structure_row["selected_candidate_id"],
            "hyperparameter_candidate_id": config_id,
            "selected_hyperparameters": str(GRIDS[model][config_id]),
            "final_fixed_epochs": frozen_epochs.get((model, target), np.nan),
            "neural_final_ensemble_seeds": "19|119|219" if model in {"MLP", "TRANSFORMER"} else "",
        })
selected_model_specifications = pd.DataFrame(selected_spec_rows).sort_values(["target", "model"]).reset_index(drop=True)
SPECIFICATIONS_FROZEN = True
display(selected_model_specifications)

Hyperparameter selection complete: RF target R


Hyperparameter selection complete: RF target I


Hyperparameter selection complete: GB target R


Hyperparameter selection complete: GB target I


Hyperparameter selection complete: MLP target R


Hyperparameter selection complete: MLP target I


Hyperparameter selection complete: TRANSFORMER target R


Hyperparameter selection complete: TRANSFORMER target I


,model,target,selected_parent_blocks,selected_lag_order,structure_candidate_id,hyperparameter_candidate_id,selected_hyperparameters,final_fixed_epochs,neural_final_ensemble_seeds
0,GB,I,I,2,I__I__L02,GB_04,"{'n_estimators': 400, 'learning_rate': 0.05}",NaN,
1,MLP,I,I|Q,4,I__I+Q__L04,MLP_01,"([32], 0.001)",59.0,19|119|219
2,RF,I,I,2,I__I__L02,RF_01,"{'max_depth': 8, 'min_samples_leaf': 5}",NaN,
3,TRANSFORMER,I,I|Q|R,5,I__I+Q+R__L05,TR_04,"((24, 4, 48), 0.0003)",61.0,19|119|219
4,GB,R,R|I,2,R__R+I__L02,GB_02,"{'n_estimators': 200, 'learning_rate': 0.05}",NaN,
5,MLP,R,R|I,1,R__R+I__L01,MLP_03,"([32, 16], 0.001)",55.0,19|119|219
6,RF,R,R|I,1,R__R+I__L01,RF_02,"{'max_depth': 8, 'min_samples_leaf': 10}",NaN,
7,TRANSFORMER,R,R|I,1,R__R+I__L01,TR_02,"((16, 2, 32), 0.0003)",148.0,19|119|219


In [8]:
validation_fitted = {}
validation_neural_ensembles = {}
validation_fit_frames = {}
for model in MODEL_FAMILIES:
    for target in TARGETS:
        parents, lag_order = selected_structure(model, target)
        columns = feature_columns(parents, lag_order)
        train_frame = model_frame(target, "train", parents, lag_order)
        validation_frame = model_frame(target, "validation", parents, lag_order)
        validation_fit_frames[(model, target)] = validation_frame
        config_id = selected_hyperparameters[(model, target)]
        if model in {"RF", "GB"}:
            estimator = make_tree(model, config_id)
            estimator.fit(train_frame[columns], train_frame[target])
            validation_fitted[(model, target)] = estimator
        else:
            networks, scales = [], []
            for seed in SEEDS:
                network, scale, _ = train_neural_early(
                    model, config_id, seed, train_frame, validation_frame,
                    target, parents, lag_order,
                )
                networks.append(network)
                scales.append(scale)
            validation_neural_ensembles[(model, target)] = (networks, scales)

def validation_prediction(model, target, frame_or_array):
    parents, lag_order = selected_structure(model, target)
    columns = feature_columns(parents, lag_order)
    if model in {"RF", "GB"}:
        values = frame_or_array[columns] if isinstance(frame_or_array, pd.DataFrame) else frame_or_array
        return validation_fitted[(model, target)].predict(values)
    networks, scales = validation_neural_ensembles[(model, target)]
    return ensemble_prediction(networks, frame_or_array, columns, scales, model, lag_order, len(parents))

stability_rows = []
for model in MODEL_FAMILIES:
    for target in TARGETS:
        selected_parents, lag_order = selected_structure(model, target)
        config_id = selected_hyperparameters[(model, target)]
        train_frame = SEARCH_SUPPORT[(target, "train")]
        validation_frame = SEARCH_SUPPORT[(target, "validation")]
        parent_results = []
        for parents in PARENT_SETS[target]:
            columns = feature_columns(parents, lag_order)
            if model in {"RF", "GB"}:
                estimator = make_tree(model, config_id)
                estimator.fit(train_frame[columns], train_frame[target])
                prediction = estimator.predict(validation_frame[columns])
                rmse = compute_metrics(validation_frame[target], prediction)["RMSE"]
            else:
                seed_rmses = []
                for seed in SEEDS:
                    network, scale, best_epoch = train_neural_early(
                        model, config_id, seed, train_frame, validation_frame,
                        target, parents, lag_order,
                    )
                    prediction = predict_neural(network, validation_frame, columns, scale, model, lag_order, len(parents))
                    rmse_seed = compute_metrics(validation_frame[target], prediction)["RMSE"]
                    neural_seed_records.append({
                        "stage": "STRUCTURE_STABILITY", "model": model, "target": target,
                        "candidate_id": "|".join(parents), "config_id": config_id, "seed": seed,
                        "parent_blocks": "|".join(parents), "lag_order": lag_order,
                        "best_epoch": best_epoch, "validation_RMSE": rmse_seed,
                        "validation_MAE": np.nan,
                    })
                    seed_rmses.append(rmse_seed)
                    del network
                rmse = float(np.mean(seed_rmses))
            parent_results.append(("|".join(parents), rmse))
        best_rmse = min(value for _, value in parent_results)
        selected_rmse = dict(parent_results)["|".join(selected_parents)]
        stability_rows.append({
            "model": model, "target": target,
            "selected_parent_blocks": "|".join(selected_parents),
            "selected_lag_order": lag_order,
            "final_hyperparameter_candidate_id": config_id,
            "best_parent_RMSE_at_selected_lag": best_rmse,
            "selected_parent_RMSE_at_selected_lag": selected_rmse,
            "selected_to_best_ratio": selected_rmse / best_rmse,
            "stable_with_final_hyperparameters": bool(selected_rmse <= PARSIMONY_BAND * best_rmse),
        })
structure_stability_check = pd.DataFrame(stability_rows)
assert len(structure_stability_check) == 8
display(structure_stability_check)

,model,target,selected_parent_blocks,selected_lag_order,final_hyperparameter_candidate_id,best_parent_RMSE_at_selected_lag,selected_parent_RMSE_at_selected_lag,selected_to_best_ratio,stable_with_final_hyperparameters
0,RF,R,R|I,1,RF_02,0.005039,0.005039,1.000000,True
1,RF,I,I,2,RF_01,0.004468,0.004490,1.004850,True
2,GB,R,R|I,2,GB_02,0.005016,0.005168,1.030288,False
3,GB,I,I,2,GB_04,0.004755,0.004755,1.000000,True
4,MLP,R,R|I,1,MLP_03,0.004913,0.004913,1.000000,True
5,MLP,I,I|Q,4,MLP_01,0.004388,0.004388,1.000000,True
6,TRANSFORMER,R,R|I,1,TR_02,0.004970,0.004972,1.000421,True
7,TRANSFORMER,I,I|Q|R,5,TR_04,0.004303,0.004372,1.016059,False


The selected specification table fixes all eight model–target designs before TEST access. Final configurations are RF_02/GB_02/MLP_03/TR_02 for R and RF_01/GB_04/MLP_01/TR_04 for I. Neural fixed epochs are 55 and 148 for R MLP and Transformer, and 59 and 61 for I. Six structures remain inside the 1% stability band under final hyperparameters; GB-R and Transformer-I are transparently flagged as probe-sensitive and are not reselected.

## 19.5 Step 2 — event-unaware forecasting

Structures, hyperparameters, and neural epoch rules are frozen. Development forecasts retain TRAIN → VALIDATION results. Final TEST forecasts refit on TRAIN+VALIDATION once, use observed lag histories for one-step-ahead prediction, and never recurse or clip. N17 is loaded as the frozen parametric benchmark and is not refit.

Model-support metrics use each discovered specification's valid dates. The primary cross-model comparison uses dates common to N17 and all four flexible models.

In [9]:
assert SPECIFICATIONS_FROZEN and STRUCTURES_FROZEN
assert selection_audit["test_used_for_selection"] is False

def frozen_fit_predict(model, target, calibration, prediction_frame):
    parents, lag_order = selected_structure(model, target)
    columns = feature_columns(parents, lag_order)
    config_id = selected_hyperparameters[(model, target)]
    if model in {"RF", "GB"}:
        estimator = make_tree(model, config_id)
        estimator.fit(calibration[columns], calibration[target])
        return estimator.predict(prediction_frame[columns])
    networks, scales = [], []
    for seed in SEEDS:
        network, scale = train_neural_fixed(
            model, config_id, seed, calibration, target, parents, lag_order,
            frozen_epochs[(model, target)],
        )
        networks.append(network)
        scales.append(scale)
    prediction = ensemble_prediction(
        networks, prediction_frame, columns, scales, model, lag_order, len(parents)
    )
    for network in networks:
        del network
    return prediction

def forecast_records(split):
    records = []
    for model in MODEL_FAMILIES:
        for target in TARGETS:
            parents, lag_order = selected_structure(model, target)
            prediction_frame = model_frame(target, split, parents, lag_order)
            if split == "validation":
                prediction = validation_prediction(model, target, prediction_frame)
                stage = "TRAIN_TO_VALIDATION"
            else:
                train_frame = model_frame(target, "train", parents, lag_order)
                validation_frame = model_frame(target, "validation", parents, lag_order)
                calibration = pd.concat([train_frame, validation_frame], ignore_index=True).sort_values("model_day")
                assert calibration["sample_split"].isin(["train", "validation"]).all()
                prediction = frozen_fit_predict(model, target, calibration, prediction_frame)
                stage = "TRAIN_VALIDATION_TO_TEST"
            for model_day, actual, value in zip(prediction_frame["model_day"], prediction_frame[target], prediction):
                records.append({
                    "model_day": model_day, "sample_split": split, "forecast_stage": stage,
                    "target": target, "model": model, "actual": float(actual),
                    "background_forecast": float(value), "is_out_of_sample": True,
                    "selected_parent_blocks": "|".join(parents), "selected_lag_order": lag_order,
                })
    return pd.DataFrame(records)

event_unaware_forecasts_validation = forecast_records("validation")
event_unaware_forecasts_test = forecast_records("test")

n17_validation = pd.read_csv(PROCESSED / "17_forecasts_validation.csv", parse_dates=["model_day"])
n17_test = pd.read_csv(PROCESSED / "17_forecasts_test.csv", parse_dates=["model_day"])
for split, source, scheme in [
    ("validation", n17_validation, "fixed_train"),
    ("test", n17_test, "fixed_train_validation"),
]:
    benchmark_parts = []
    for target, model_name in [("R", "R_PRIMARY_AR1_IV"), ("I", "I_PRIMARY_FULL_12")]:
        part = source.loc[
            source["target"].eq(target) & source["model"].eq(model_name) & source["forecast_scheme"].eq(scheme),
            ["model_day", "actual", "raw_forecast"],
        ].copy()
        part["sample_split"] = split
        part["forecast_stage"] = "FROZEN_N17_" + scheme.upper()
        part["target"] = target
        part["model"] = "PARAMETRIC_N17"
        part["background_forecast"] = part.pop("raw_forecast")
        part["is_out_of_sample"] = True
        part["selected_parent_blocks"] = "N16_FROZEN"
        part["selected_lag_order"] = int(spec_16["selected_realised_order"] if target == "R" else spec_16["selected_implied_order"])
        benchmark_parts.append(part)
    benchmark = pd.concat(benchmark_parts, ignore_index=True)
    if split == "validation":
        event_unaware_forecasts_validation = pd.concat([event_unaware_forecasts_validation, benchmark], ignore_index=True)
    else:
        event_unaware_forecasts_test = pd.concat([event_unaware_forecasts_test, benchmark], ignore_index=True)

def metric_table(forecasts, support):
    rows = []
    for (target, model), group in forecasts.groupby(["target", "model"], sort=True):
        lookup = panel.set_index("model_day")
        persistence = lookup.loc[group["model_day"], f"{target}_lag1"].to_numpy(dtype=float)
        metric = compute_metrics(group["actual"], group["background_forecast"], persistence)
        rows.append({"support": support, "target": target, "model": model, **metric})
    return pd.DataFrame(rows)

forecast_metrics_validation = metric_table(event_unaware_forecasts_validation, "MODEL_SPECIFIC")
forecast_metrics_test_model_support = metric_table(event_unaware_forecasts_test, "MODEL_SPECIFIC")

common_parts = []
for target in TARGETS:
    target_rows = event_unaware_forecasts_test.loc[event_unaware_forecasts_test["target"].eq(target)]
    models = set(target_rows["model"])
    assert models == set((*MODEL_FAMILIES, "PARAMETRIC_N17"))
    common_dates = None
    for model in models:
        dates = set(target_rows.loc[
            target_rows["model"].eq(model) & np.isfinite(target_rows["background_forecast"]), "model_day"
        ])
        common_dates = dates if common_dates is None else common_dates & dates
    common_parts.append(target_rows.loc[target_rows["model_day"].isin(common_dates)].copy())
event_unaware_test_common = pd.concat(common_parts, ignore_index=True)
forecast_metrics_test_common_support = metric_table(event_unaware_test_common, "COMMON_ALL_MODELS")
assert forecast_metrics_test_common_support.groupby("target")["valid_forecast_n"].nunique().eq(1).all()

fig, ax = plt.subplots(figsize=(10, 5.5), constrained_layout=True)
plot = forecast_metrics_test_common_support.copy()
models = ["PARAMETRIC_N17", *MODEL_FAMILIES]
x = np.arange(len(models))
width = 0.35
for offset, target in [(-width / 2, "R"), (width / 2, "I")]:
    rows = plot.loc[plot["target"].eq(target)].set_index("model").reindex(models)
    ax.bar(x + offset, rows["RMSE"], width=width, label=target)
ax.set_xticks(x, models, rotation=25, ha="right")
ax.set_ylabel("TEST RMSE")
ax.set_title("Figure 19.4. TEST RMSE on common support")
ax.legend(title="Target")
save_figure(fig, "19_figure_19_4_test_rmse_comparison.png")

for target, filename, label in [
    ("R", "19_figure_19_5_R_test_forecasts.png", "realised variance"),
    ("I", "19_figure_19_6_I_test_forecasts.png", "implied variance"),
]:
    candidates = forecast_metrics_validation.loc[
        forecast_metrics_validation["target"].eq(target) & forecast_metrics_validation["model"].isin(["RF", "GB", "MLP"])
    ]
    best_non_transformer = candidates.sort_values(["RMSE", "model"]).iloc[0]["model"]
    chosen = ["PARAMETRIC_N17", "TRANSFORMER", best_non_transformer]
    target_rows = event_unaware_forecasts_test.loc[
        event_unaware_forecasts_test["target"].eq(target) & event_unaware_forecasts_test["model"].isin(chosen)
    ]
    common_dates = set.intersection(*[
        set(target_rows.loc[target_rows["model"].eq(model), "model_day"]) for model in chosen
    ])
    target_rows = target_rows.loc[target_rows["model_day"].isin(common_dates)]
    actual = target_rows.loc[target_rows["model"].eq(chosen[0]), ["model_day", "actual"]].sort_values("model_day")
    fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)
    ax.plot(actual["model_day"], actual["actual"], color="black", linewidth=1.0, alpha=0.65, label=f"Actual {target}")
    styles = {
        "PARAMETRIC_N17": ("#555555", 1.2),
        "TRANSFORMER": ("#A64D79", 1.2),
        best_non_transformer: ("#4472C4", 1.1),
    }
    for model in chosen:
        rows = target_rows.loc[target_rows["model"].eq(model)].sort_values("model_day")
        color, linewidth = styles[model]
        ax.plot(rows["model_day"], rows["background_forecast"], color=color, linewidth=linewidth, alpha=0.9, label=model)
    ax.set_title(f"Figure 19.{5 if target == 'R' else 6}. TEST {label}: actual and frozen forecasts")
    ax.set_ylabel(target)
    ax.legend(ncol=4)
    save_figure(fig, filename)

print("Common-support TEST metrics")
display(forecast_metrics_test_common_support)

Common-support TEST metrics


,support,target,model,valid_forecast_n,RMSE,MAE,QLIKE,QLIKE_valid_n,OOS_R2,nonpositive_forecast_n
0,COMMON_ALL_MODELS,I,GB,1071,0.011889,0.004853,0.138120,1071,0.378009,0
1,COMMON_ALL_MODELS,I,MLP,1071,0.011826,0.004871,0.155322,1070,0.384565,1
2,COMMON_ALL_MODELS,I,PARAMETRIC_N17,1071,0.013857,0.006476,0.622338,1046,0.155036,25
3,COMMON_ALL_MODELS,I,RF,1071,0.012116,0.004877,0.133424,1071,0.354016,0
4,COMMON_ALL_MODELS,I,TRANSFORMER,1071,0.011452,0.004426,0.144487,1071,0.422905,0
5,COMMON_ALL_MODELS,R,GB,1169,0.015227,0.005613,0.317121,1169,0.490031,0
6,COMMON_ALL_MODELS,R,MLP,1169,0.014824,0.005416,0.298315,1169,0.516636,0
7,COMMON_ALL_MODELS,R,PARAMETRIC_N17,1169,0.015107,0.005582,0.329447,1169,0.498001,0
8,COMMON_ALL_MODELS,R,RF,1169,0.015130,0.005500,0.313422,1169,0.496488,0
9,COMMON_ALL_MODELS,R,TRANSFORMER,1169,0.014983,0.005333,0.303804,1169,0.506225,0


The TEST comparison above is descriptive and was produced only after every model decision froze. On common R support, MLP has the lowest RMSE (0.014824) versus 0.015107 for N17, with MAE 0.005416, QLIKE 0.298315, and OOS R² 0.516636; Transformer also improves RMSE to 0.014983, whereas RF and GB are slightly worse than N17. On common I support, Transformer is best (RMSE 0.011452, MAE 0.004426, QLIKE 0.144487, OOS R² 0.422905) and every flexible model beats N17's RMSE 0.013857. Among flexible common-support forecasts, only I-MLP has a non-positive value (one); N17 has 25 for I. The plotted non-Transformer comparator was chosen by VALIDATION RMSE, never by TEST performance.

## 19.6 Post-fit predictive-reliance diagnostics

Permutation importance was **not** used to discover structure. Structure was selected earlier by conditional out-of-sample candidate comparison. The diagnostics below measure post-fit predictive reliance on ordinary VALIDATION data only.

Block effects jointly permute a retained source history. Lag effects jointly permute retained variables at one lag. Transformer heatmaps show individual selected variable–lag effects; they do not display attention weights or causal edges.

In [10]:
PERMUTATION_REPEATS = 20
TRANSFORMER_FEATURE_REPEATS = 30
importance_rows = []
rng = np.random.default_rng(19)

for model in MODEL_FAMILIES:
    for target in TARGETS:
        parents, lag_order = selected_structure(model, target)
        columns = feature_columns(parents, lag_order)
        frame = validation_fit_frames[(model, target)]
        original = frame[columns].to_numpy(dtype=float)
        actual = frame[target].to_numpy(dtype=float)
        baseline = compute_metrics(actual, validation_prediction(model, target, original))["RMSE"]

        for state in parents:
            indices = [index for index, column in enumerate(columns) if column.startswith(state + "_lag")]
            deltas = []
            for _ in range(PERMUTATION_REPEATS):
                permutation = rng.permutation(len(frame))
                permuted = original.copy()
                permuted[:, indices] = original[permutation][:, indices]
                rmse = compute_metrics(actual, validation_prediction(model, target, permuted))["RMSE"]
                deltas.append(rmse - baseline)
            importance_rows.append({
                "target": target, "model": model, "importance_scope": "PARENT_BLOCK",
                "parent_block": state, "lag": np.nan, "feature": state + "_BLOCK",
                "n_repeats": len(deltas), "baseline_RMSE": baseline,
                "mean_delta_RMSE": float(np.mean(deltas)),
                "median_delta_RMSE": float(np.median(deltas)),
                "validation_only": True,
            })

        for lag in range(1, lag_order + 1):
            indices = [index for index, column in enumerate(columns) if column.endswith(f"_lag{lag}")]
            deltas = []
            for _ in range(PERMUTATION_REPEATS):
                permutation = rng.permutation(len(frame))
                permuted = original.copy()
                permuted[:, indices] = original[permutation][:, indices]
                rmse = compute_metrics(actual, validation_prediction(model, target, permuted))["RMSE"]
                deltas.append(rmse - baseline)
            importance_rows.append({
                "target": target, "model": model, "importance_scope": "LAG_GROUP",
                "parent_block": "|".join(parents), "lag": lag, "feature": f"LAG_{lag}",
                "n_repeats": len(deltas), "baseline_RMSE": baseline,
                "mean_delta_RMSE": float(np.mean(deltas)),
                "median_delta_RMSE": float(np.median(deltas)),
                "validation_only": True,
            })

        if model == "TRANSFORMER":
            for index, column in enumerate(columns):
                state, lag_text = column.split("_lag")
                deltas = []
                for _ in range(TRANSFORMER_FEATURE_REPEATS):
                    permutation = rng.permutation(len(frame))
                    permuted = original.copy()
                    permuted[:, index] = original[permutation, index]
                    rmse = compute_metrics(actual, validation_prediction(model, target, permuted))["RMSE"]
                    deltas.append(rmse - baseline)
                importance_rows.append({
                    "target": target, "model": model, "importance_scope": "INDIVIDUAL_FEATURE",
                    "parent_block": state, "lag": int(lag_text), "feature": column,
                    "n_repeats": len(deltas), "baseline_RMSE": baseline,
                    "mean_delta_RMSE": float(np.mean(deltas)),
                    "median_delta_RMSE": float(np.median(deltas)),
                    "validation_only": True,
                })

predictive_relevance_validation = pd.DataFrame(importance_rows)
assert predictive_relevance_validation["validation_only"].all()

for target, filename, number in [
    ("R", "19_figure_19_7_transformer_predictive_relevance_R.png", 7),
    ("I", "19_figure_19_8_transformer_predictive_relevance_I.png", 8),
]:
    parents, lag_order = selected_structure("TRANSFORMER", target)
    rows = predictive_relevance_validation.loc[
        predictive_relevance_validation["target"].eq(target)
        & predictive_relevance_validation["model"].eq("TRANSFORMER")
        & predictive_relevance_validation["importance_scope"].eq("INDIVIDUAL_FEATURE")
    ]
    matrix = rows.pivot(index="parent_block", columns="lag", values="mean_delta_RMSE").reindex(
        index=list(parents), columns=list(range(lag_order, 0, -1))
    )
    limit = max(float(np.nanmax(np.abs(matrix.to_numpy()))), 1e-12)
    fig, ax = plt.subplots(figsize=(max(7, lag_order * 0.75), 2.8 + len(parents)), constrained_layout=True)
    image = ax.imshow(matrix.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-limit, vmax=limit)
    ax.set_xticks(range(lag_order), [f"lag {lag}" for lag in range(lag_order, 0, -1)], rotation=45, ha="right")
    ax.set_yticks(range(len(parents)), parents)
    ax.set_title(f"Figure 19.{number}. Transformer predictive relevance for target {target}")
    ax.set_xlabel("Selected lag")
    ax.set_ylabel("Selected variable")
    fig.colorbar(image, ax=ax, label="Mean delta VALIDATION RMSE")
    save_figure(fig, filename)

display(predictive_relevance_validation.loc[predictive_relevance_validation["importance_scope"].eq("PARENT_BLOCK")])

C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


...

C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\python

C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
C:\Users\Rajiv Nawal\AppData\Local\Pytho

,target,model,importance_scope,parent_block,lag,feature,n_repeats,baseline_RMSE,mean_delta_RMSE,median_delta_RMSE,validation_only
0,R,RF,PARENT_BLOCK,R,NaN,R_BLOCK,20,0.006241,0.000491,0.000489,True
1,R,RF,PARENT_BLOCK,I,NaN,I_BLOCK,20,0.006241,0.002001,0.002042,True
3,I,RF,PARENT_BLOCK,I,NaN,I_BLOCK,20,0.004501,0.006030,0.006029,True
6,R,GB,PARENT_BLOCK,R,NaN,R_BLOCK,20,0.004955,0.000294,0.000289,True
7,R,GB,PARENT_BLOCK,I,NaN,I_BLOCK,20,0.004955,0.002937,0.002977,True
10,I,GB,PARENT_BLOCK,I,NaN,I_BLOCK,20,0.005183,0.005218,0.005230,True
13,R,MLP,PARENT_BLOCK,R,NaN,R_BLOCK,20,0.006145,0.000209,0.000242,True
14,R,MLP,PARENT_BLOCK,I,NaN,I_BLOCK,20,0.006145,0.002693,0.002725,True
16,I,MLP,PARENT_BLOCK,I,NaN,I_BLOCK,20,0.004192,0.005953,0.005963,True
17,I,MLP,PARENT_BLOCK,Q,NaN,Q_BLOCK,20,0.004192,0.000484,0.000499,True


The relevance tables are conditional on each already-selected model. Positive delta RMSE indicates that shuffling a retained block or lag worsened VALIDATION forecasts; negative values indicate that the fitted model did not benefit reliably from that component in this diagnostic.

## 19.7 Step 3 — eventless refitting with frozen specifications

Corrected N18 masks enter for the first time here. Parent blocks, lag orders, hyperparameters, seeds, scaling method, and neural epoch counts remain unchanged.

For VALIDATION backgrounds, event-age-0 target observations are removed only from TRAIN fitting targets. For TEST backgrounds, they are removed only from TRAIN+VALIDATION fitting targets. The chronology and lagged Q/R/I histories remain intact, so observed event states can legitimately appear as later predictors.

In [11]:
assert SPECIFICATIONS_FROZEN and STRUCTURES_FROZEN
spec_18 = pd.read_csv(PROCESSED / "18_primary_specification.csv").iloc[0]
event_flags = pd.read_csv(PROCESSED / "18_event_day_flags.csv", parse_dates=["model_day"])
target_specific_mapping = pd.read_csv(
    PROCESSED / "18_target_specific_event_mapping.csv",
    parse_dates=["realised_model_day", "implied_model_day", "target_model_day", "model_day", "event_timestamp_utc"],
)
n18_event_weights = pd.read_csv(
    PROCESSED / "18_event_response_weights.csv",
    parse_dates=["realised_model_day", "implied_model_day", "target_model_day", "model_day", "event_timestamp_utc"],
)

assert str(spec_18["event_alignment_version"]) == "BLOOMBERG_BGN_1700_TO_1659_SAME_MODEL_DAY"
assert as_bool(spec_18["bloomberg_pricing_session_identified"])
assert not as_bool(spec_18["bloomberg_snapshot_time_identified"])
assert not as_bool(spec_18["approximate_10am_iv_cut_used_for_production"])
assert not as_bool(spec_18["linus_equation_5_used"])
assert target_specific_mapping.loc[target_specific_mapping["target"].eq("R"), "target_model_day"].eq(
    target_specific_mapping.loc[target_specific_mapping["target"].eq("R"), "realised_model_day"]
).all()
assert target_specific_mapping.loc[target_specific_mapping["target"].eq("I"), "target_model_day"].eq(
    target_specific_mapping.loc[target_specific_mapping["target"].eq("I"), "implied_model_day"]
).all()
assert target_specific_mapping["realised_model_day"].eq(target_specific_mapping["implied_model_day"]).all()

flag_columns = [
    "model_day",
    "R_is_target_event_day", "R_is_clean_single_family", "R_is_overlap",
    "R_n_target_occurrences", "R_n_target_families", "R_family_set",
    "I_is_target_event_day", "I_is_clean_single_family", "I_is_overlap",
    "I_n_target_occurrences", "I_n_target_families", "I_family_set",
]
panel = panel.merge(event_flags[flag_columns], on="model_day", how="left", validate="one_to_one")
for target in TARGETS:
    for suffix in ["is_target_event_day", "is_clean_single_family", "is_overlap"]:
        panel[f"{target}_{suffix}"] = panel[f"{target}_{suffix}"].eq(True)
    for suffix in ["n_target_occurrences", "n_target_families"]:
        panel[f"{target}_{suffix}"] = pd.to_numeric(panel[f"{target}_{suffix}"], errors="coerce").fillna(0).astype(int)
    panel[f"{target}_family_set"] = panel[f"{target}_family_set"].fillna("")
assert panel[["model_day", *MAX_LAG_FEATURES]].equals(lag_snapshot)

for target in TARGETS:
    event_indices = panel.index[panel[f"{target}_is_target_event_day"]]
    event_indices = event_indices[event_indices < len(panel) - 1]
    for state in STATES:
        previous = panel.loc[event_indices, state].to_numpy(dtype=float)
        following_lag = panel.loc[event_indices + 1, f"{state}_lag1"].to_numpy(dtype=float)
        finite = np.isfinite(previous) & np.isfinite(following_lag)
        assert np.allclose(previous[finite], following_lag[finite])

def eventless_calibration(target, splits, parents, lag_order):
    frames = [model_frame(target, split, parents, lag_order) for split in splits]
    ordinary = pd.concat(frames, ignore_index=True).sort_values("model_day")
    eventless = ordinary.loc[~ordinary[f"{target}_is_target_event_day"]].copy()
    return ordinary, eventless

def build_eventless_forecasts(prediction_split):
    records = []
    audits = []
    calibration_splits = ("train",) if prediction_split == "validation" else ("train", "validation")
    stage = "EVENTLESS_TRAIN_TO_VALIDATION" if prediction_split == "validation" else "EVENTLESS_TRAIN_VALIDATION_TO_TEST"
    for model in MODEL_FAMILIES:
        for target in TARGETS:
            parents, lag_order = selected_structure(model, target)
            ordinary_fit, eventless_fit = eventless_calibration(target, calibration_splits, parents, lag_order)
            prediction_frame = model_frame(target, prediction_split, parents, lag_order)
            prediction = frozen_fit_predict(model, target, eventless_fit, prediction_frame)
            audits.append({
                "model": model, "target": target, "forecast_stage": stage,
                "calibration_splits": "|".join(calibration_splits),
                "ordinary_eligible_n": len(ordinary_fit),
                "removed_target_events_n": int(ordinary_fit[f"{target}_is_target_event_day"].sum()),
                "eventless_fit_n": len(eventless_fit),
                "prediction_n": len(prediction_frame),
                "structure_frozen": True, "hyperparameters_frozen": True,
            })
            for row, value in zip(prediction_frame.itertuples(index=False), prediction):
                records.append({
                    "model_day": row.model_day, "target_model_day": row.model_day,
                    "sample_split": prediction_split, "forecast_stage": stage,
                    "target": target, "model": model, "actual": float(getattr(row, target)),
                    "background_forecast": float(value), "is_out_of_sample": True,
                    "is_target_event_day": bool(getattr(row, f"{target}_is_target_event_day")),
                    "is_clean_single_family": bool(getattr(row, f"{target}_is_clean_single_family")),
                    "is_overlap": bool(getattr(row, f"{target}_is_overlap")),
                    "n_target_occurrences": int(getattr(row, f"{target}_n_target_occurrences")),
                    "n_target_families": int(getattr(row, f"{target}_n_target_families")),
                    "family_set": getattr(row, f"{target}_family_set"),
                })
    return pd.DataFrame(records), pd.DataFrame(audits)

eventless_forecasts_validation, eventless_audit_validation = build_eventless_forecasts("validation")
eventless_forecasts_test, eventless_audit_test = build_eventless_forecasts("test")
eventless_support_audit = pd.concat([eventless_audit_validation, eventless_audit_test], ignore_index=True)
assert eventless_support_audit["structure_frozen"].all()
assert eventless_support_audit["hyperparameters_frozen"].all()
assert (eventless_support_audit["ordinary_eligible_n"] - eventless_support_audit["removed_target_events_n"] == eventless_support_audit["eventless_fit_n"]).all()
display(eventless_support_audit)

,model,target,forecast_stage,calibration_splits,ordinary_eligible_n,removed_target_events_n,eventless_fit_n,prediction_n,structure_frozen,hyperparameters_frozen
0,RF,R,EVENTLESS_TRAIN_TO_VALIDATION,train,3005,684,2321,1181,True,True
1,RF,I,EVENTLESS_TRAIN_TO_VALIDATION,train,3015,688,2327,1209,True,True
2,GB,R,EVENTLESS_TRAIN_TO_VALIDATION,train,2782,641,2141,1168,True,True
3,GB,I,EVENTLESS_TRAIN_TO_VALIDATION,train,3015,688,2327,1209,True,True
4,MLP,R,EVENTLESS_TRAIN_TO_VALIDATION,train,3005,684,2321,1181,True,True
5,MLP,I,EVENTLESS_TRAIN_TO_VALIDATION,train,2554,591,1963,1155,True,True
6,TRANSFORMER,R,EVENTLESS_TRAIN_TO_VALIDATION,train,3005,684,2321,1181,True,True
7,TRANSFORMER,I,EVENTLESS_TRAIN_TO_VALIDATION,train,2497,575,1922,1143,True,True
8,RF,R,EVENTLESS_TRAIN_VALIDATION_TO_TEST,train|validation,4186,946,3240,1182,True,True
9,RF,I,EVENTLESS_TRAIN_VALIDATION_TO_TEST,train|validation,4224,952,3272,1208,True,True


## 19.8 Ordinary ratios, clean TEST event weights, and robustness

A response ratio is observed variance divided by the raw background forecast. It is defined only when both quantities are finite and the denominator is strictly positive. No clipping, epsilon, absolute value, normalisation, or positivity transform is applied.

The clean TEST comparison uses N18's single-family event definition and adds the authoritative N18 parametric benchmark without refitting it.

In [12]:
ordinary_response_ratios = pd.concat(
    [event_unaware_forecasts_validation, event_unaware_forecasts_test],
    ignore_index=True,
)
ordinary_response_ratios["target_model_day"] = ordinary_response_ratios["model_day"]
for target in TARGETS:
    flag_lookup = panel.set_index("model_day")[f"{target}_is_target_event_day"]
    mask = ordinary_response_ratios["target"].eq(target)
    ordinary_response_ratios.loc[mask, "known_event_target_flag"] = ordinary_response_ratios.loc[mask, "model_day"].map(flag_lookup)
ordinary_response_ratios["known_event_target_flag"] = ordinary_response_ratios["known_event_target_flag"].eq(True)
ordinary_response_ratios["ordinary_flag"] = ~ordinary_response_ratios["known_event_target_flag"]
ordinary_response_ratios["equation_eligible"] = np.isfinite(ordinary_response_ratios["actual"]) & np.isfinite(ordinary_response_ratios["background_forecast"])
ordinary_response_ratios["valid_positive_background"] = ordinary_response_ratios["equation_eligible"] & ordinary_response_ratios["background_forecast"].gt(0)
ordinary_response_ratios["response_ratio"] = np.where(
    ordinary_response_ratios["valid_positive_background"],
    ordinary_response_ratios["actual"] / ordinary_response_ratios["background_forecast"],
    np.nan,
)
ordinary_response_ratios = ordinary_response_ratios[[
    "model_day", "target_model_day", "sample_split", "forecast_stage", "target", "model",
    "actual", "background_forecast", "response_ratio", "valid_positive_background",
    "equation_eligible", "known_event_target_flag", "ordinary_flag", "is_out_of_sample",
]]

mapping_columns = [
    "event_id", "event_label", "family", "event_timestamp_utc", "event_timestamp_ny",
    "realised_model_day", "implied_model_day", "target", "target_model_day",
    "sample_split", "is_clean_single_family", "is_overlap", "n_target_occurrences",
]
test_mapping = target_specific_mapping.loc[target_specific_mapping["sample_split"].eq("test"), mapping_columns].copy()
assert set(test_mapping["family"]) == set(FAMILIES)
flexible_event_parts = []
for target in TARGETS:
    occurrences = test_mapping.loc[test_mapping["target"].eq(target)].copy()
    model_table = pd.DataFrame({"model": MODEL_FAMILIES})
    occurrences["_key"] = 1
    model_table["_key"] = 1
    cartesian = occurrences.merge(model_table, on="_key").drop(columns="_key")
    forecasts = eventless_forecasts_test.loc[
        eventless_forecasts_test["target"].eq(target),
        ["target_model_day", "target", "model", "actual", "background_forecast", "forecast_stage", "is_out_of_sample"],
    ]
    cartesian = cartesian.merge(
        forecasts, on=["target_model_day", "target", "model"], how="left", validate="many_to_one"
    )
    actual_lookup = panel.set_index("model_day")[target]
    cartesian["actual"] = cartesian["actual"].fillna(cartesian["target_model_day"].map(actual_lookup))
    cartesian["forecast_stage"] = cartesian["forecast_stage"].fillna("EVENTLESS_TRAIN_VALIDATION_TO_TEST")
    cartesian["is_out_of_sample"] = True
    cartesian["equation_eligible"] = np.isfinite(cartesian["actual"]) & np.isfinite(cartesian["background_forecast"])
    cartesian["background_positive"] = cartesian["equation_eligible"] & cartesian["background_forecast"].gt(0)
    cartesian["weight_defined"] = cartesian["background_positive"]
    cartesian["response_ratio"] = np.where(
        cartesian["weight_defined"], cartesian["actual"] / cartesian["background_forecast"], np.nan
    )
    cartesian["event_weight"] = cartesian["response_ratio"]
    cartesian["undefined_reason"] = np.select(
        [~cartesian["equation_eligible"], cartesian["equation_eligible"] & ~cartesian["background_positive"]],
        ["not_equation_eligible", "nonpositive_background"], default="",
    )
    flexible_event_parts.append(cartesian)
flexible_event_weights = pd.concat(flexible_event_parts, ignore_index=True)

parametric = n18_event_weights.loc[n18_event_weights["sample_split"].eq("test")].copy()
parametric["model"] = "PARAMETRIC_N18"
parametric["background_forecast"] = parametric["raw_background_forecast"]
parametric["event_weight"] = parametric["response_ratio"]
parametric["forecast_stage"] = "AUTHORITATIVE_N18_EVENTLESS_TEST"
parametric["is_out_of_sample"] = True
parametric["undefined_reason"] = parametric["undefined_reason"].fillna("")

event_weight_columns = [
    *mapping_columns, "model", "actual", "background_forecast", "forecast_stage",
    "is_out_of_sample", "equation_eligible", "background_positive", "weight_defined",
    "response_ratio", "event_weight", "undefined_reason",
]
event_response_weights = pd.concat(
    [parametric[event_weight_columns], flexible_event_weights[event_weight_columns]],
    ignore_index=True,
)
assert event_response_weights["sample_split"].eq("test").all()
defined = event_response_weights["weight_defined"].eq(True)
assert np.allclose(
    event_response_weights.loc[defined, "event_weight"],
    event_response_weights.loc[defined, "actual"] / event_response_weights.loc[defined, "background_forecast"],
    rtol=1e-12, atol=1e-14,
)
assert event_response_weights.loc[
    event_response_weights["equation_eligible"].eq(True) & ~event_response_weights["background_positive"].eq(True),
    "response_ratio",
].isna().all()

summary_rows = []
clean = event_response_weights.loc[event_response_weights["is_clean_single_family"].eq(True)].copy()
for (family, target, model), group in clean.groupby(["family", "target", "model"], sort=True):
    valid = group.loc[group["weight_defined"].eq(True), "response_ratio"].to_numpy(dtype=float)
    summary_rows.append({
        "family": family, "target": target, "model": model,
        "clean_occurrence_n": len(group),
        "equation_eligible_n": int(group["equation_eligible"].eq(True).sum()),
        "valid_positive_background_n": len(valid),
        "median_event_weight": float(np.median(valid)) if len(valid) else np.nan,
        "Q25_event_weight": float(np.quantile(valid, 0.25)) if len(valid) else np.nan,
        "Q75_event_weight": float(np.quantile(valid, 0.75)) if len(valid) else np.nan,
        "mean_event_weight": float(np.mean(valid)) if len(valid) else np.nan,
        "nonpositive_background_n": int((group["equation_eligible"].eq(True) & ~group["background_positive"].eq(True)).sum()),
    })
event_weight_summary_clean_test = pd.DataFrame(summary_rows)
assert set(event_weight_summary_clean_test["model"]) == set((*MODEL_FAMILIES, "PARAMETRIC_N18"))

agreement_rows = []
benchmark = clean.loc[clean["model"].eq("PARAMETRIC_N18") & clean["weight_defined"].eq(True)]
for target in TARGETS:
    for family in FAMILIES:
        reference = benchmark.loc[
            benchmark["target"].eq(target) & benchmark["family"].eq(family),
            ["event_id", "response_ratio"],
        ].rename(columns={"response_ratio": "parametric_weight"})
        for model in MODEL_FAMILIES:
            comparator = clean.loc[
                clean["target"].eq(target) & clean["family"].eq(family)
                & clean["model"].eq(model) & clean["weight_defined"].eq(True),
                ["event_id", "response_ratio"],
            ].rename(columns={"response_ratio": "flexible_weight"})
            matched = reference.merge(comparator, on="event_id", how="inner", validate="one_to_one")
            parametric_median = float(np.median(matched["parametric_weight"])) if len(matched) else np.nan
            flexible_median = float(np.median(matched["flexible_weight"])) if len(matched) else np.nan
            agreement_rows.append({
                "target": target, "family": family, "benchmark_model": "PARAMETRIC_N18",
                "comparator_model": model, "valid_common_n": len(matched),
                "pearson_correlation": matched["parametric_weight"].corr(matched["flexible_weight"], method="pearson") if len(matched) >= 2 else np.nan,
                "spearman_correlation": matched["parametric_weight"].corr(matched["flexible_weight"], method="spearman") if len(matched) >= 2 else np.nan,
                "median_absolute_difference": float(np.median(np.abs(matched["flexible_weight"] - matched["parametric_weight"]))) if len(matched) else np.nan,
                "parametric_family_median": parametric_median,
                "flexible_family_median": flexible_median,
                "family_median_same_side_of_one": bool((parametric_median > 1) == (flexible_median > 1)) if len(matched) else False,
            })
event_weight_agreement = pd.DataFrame(agreement_rows)

model_order = ["PARAMETRIC_N18", *MODEL_FAMILIES]
colors = ["#555555", "#4472C4", "#70AD47", "#ED7D31", "#A64D79"]
for target, filename, number in [
    ("R", "19_figure_19_9_event_weight_medians_R.png", 9),
    ("I", "19_figure_19_10_event_weight_medians_I.png", 10),
]:
    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    target_summary = event_weight_summary_clean_test.loc[event_weight_summary_clean_test["target"].eq(target)]
    x = np.arange(len(FAMILIES))
    for model, color in zip(model_order, colors):
        rows = target_summary.loc[target_summary["model"].eq(model)].set_index("family").reindex(FAMILIES)
        ax.plot(x, rows["median_event_weight"], marker="o", linewidth=1.5, color=color, label=model)
    ax.axhline(1.0, color="black", linewidth=1, linestyle="--")
    ax.set_xticks(x, FAMILIES, rotation=25, ha="right")
    ax.set_ylabel("Median clean TEST event weight")
    ax.set_title(f"Figure 19.{number}. Event-weight robustness for target {target}")
    ax.legend(ncol=3)
    save_figure(fig, filename)

display(event_weight_summary_clean_test)
display(event_weight_agreement)

,family,target,model,clean_occurrence_n,equation_eligible_n,valid_positive_background_n,median_event_weight,Q25_event_weight,Q75_event_weight,mean_event_weight,nonpositive_background_n
0,BoJ,I,GB,27,27,27,0.456327,0.341397,0.776459,0.577918,0
1,BoJ,I,MLP,27,26,26,0.544481,0.442310,1.017853,0.684980,0
2,BoJ,I,PARAMETRIC_N18,27,26,26,0.541835,0.413015,0.949720,0.676932,0
3,BoJ,I,RF,27,27,27,0.518052,0.422804,0.964954,0.676678,0
4,BoJ,I,TRANSFORMER,27,26,26,0.524906,0.395119,0.935480,0.668628,0
5,BoJ,R,GB,27,26,26,0.861520,0.529668,1.388356,1.681854,0
6,BoJ,R,MLP,27,27,27,0.810789,0.601136,1.294125,1.498866,0
7,BoJ,R,PARAMETRIC_N18,27,27,27,0.840995,0.623537,1.210719,1.609328,0
8,BoJ,R,RF,27,27,27,0.894051,0.565697,1.497435,1.756103,0
9,BoJ,R,TRANSFORMER,27,27,27,0.764537,0.599552,1.263297,1.600116,0


,target,family,benchmark_model,comparator_model,valid_common_n,pearson_correlation,spearman_correlation,median_absolute_difference,parametric_family_median,flexible_family_median,family_median_same_side_of_one
0,R,FOMC,PARAMETRIC_N18,RF,33,0.984061,0.947527,0.190171,1.096284,1.169078,True
1,R,FOMC,PARAMETRIC_N18,GB,33,0.942084,0.969251,0.164046,1.096284,1.229085,True
2,R,FOMC,PARAMETRIC_N18,MLP,33,0.991273,0.967580,0.143257,1.096284,1.205821,True
3,R,FOMC,PARAMETRIC_N18,TRANSFORMER,33,0.989379,0.971925,0.109241,1.096284,1.120955,True
4,R,US Employment,PARAMETRIC_N18,RF,52,0.971535,0.963289,0.107951,1.043425,1.023048,True
5,R,US Employment,PARAMETRIC_N18,GB,52,0.987371,0.981303,0.093145,1.043425,1.009420,True
6,R,US Employment,PARAMETRIC_N18,MLP,52,0.987188,0.988218,0.063227,1.043425,0.929021,False
7,R,US Employment,PARAMETRIC_N18,TRANSFORMER,52,0.995021,0.990096,0.110881,1.043425,0.953392,False
8,R,US CPI,PARAMETRIC_N18,RF,51,0.823894,0.949050,0.106184,0.972722,0.950061,True
9,R,US CPI,PARAMETRIC_N18,GB,51,0.995880,0.982986,0.085337,0.972722,1.052252,False


The event-weight tables compare like-for-like clean TEST occurrences. Agreement statistics are computed only where both the N18 benchmark and a flexible model have valid positive backgrounds. Median direction is unanimous for 9 of 12 target-family comparisons: R is robust for FOMC, BoJ, JPY National CPI, and JPY GDP but not US Employment or US CPI; I is robust for every family except FOMC. Flexible event backgrounds produce one non-positive I value for GB and one for MLP; RF and Transformer produce none. The loaded N18 benchmark contributes three non-positive I event backgrounds. No denominator is repaired.

## 19.9 Authoritative exports and freeze audits

The manifest below contains only objects required to reproduce structure selection, frozen forecasting, predictive-reliance diagnostics, eventless refits, and downstream event-weight use. Obsolete N19-only CSVs and figures are removed with prefix and parent-directory guards; no upstream or downstream artifact is eligible for deletion.

In [13]:
assert STRUCTURES_FROZEN and SPECIFICATIONS_FROZEN
assert len(structure_search_validation) == 384
assert discovered_structures["selected_lag_order"].between(1, 12).all()
assert selection_audit["test_used_for_selection"] is False
assert selection_audit["n16_structure_used_for_selection"] is False
assert selection_audit["event_mask_used_for_structure_selection"] is False
assert selection_audit["event_mask_used_for_hyperparameter_selection"] is False

for target in TARGETS:
    mapped_days = set(target_specific_mapping.loc[target_specific_mapping["target"].eq(target), "target_model_day"])
    flagged_days = set(panel.loc[panel[f"{target}_is_target_event_day"], "model_day"])
    assert mapped_days == flagged_days

neural_seed_diagnostics = pd.DataFrame(neural_seed_records)
run_metadata = pd.DataFrame([{
    "RUN_ID": RUN_ID,
    "exercise": "EXERCISE_4_NONPARAMETRIC_STEPS_1_TO_3",
    "structure_selection_split": "TRAIN_TO_VALIDATION",
    "hyperparameter_selection_split": "TRAIN_TO_VALIDATION",
    "test_used_for_selection": False,
    "n16_structure_used_for_nonparametric_selection": False,
    "event_mask_used_for_structure_selection": False,
    "event_mask_used_for_hyperparameter_selection": False,
    "permutation_importance_used_for_structure_selection": False,
    "permutation_importance_split": "VALIDATION_ONLY",
    "l_max": L_MAX,
    "lag_order_is_data_selected": True,
    "structure_candidate_count": len(structure_search_validation),
    "selected_structure_count": len(discovered_structures),
    "structures_frozen_before_n16_comparison": True,
    "structures_frozen_before_eventless_fitting": True,
    "corrected_n18_timing_inherited": True,
    "bloomberg_pricing_session_identified": True,
    "bloomberg_snapshot_time_identified": False,
    "approximate_10am_iv_cut_used_for_production": False,
    "linus_equation_5_used": False,
    "neural_seeds": "19|119|219",
    "neural_epoch_rule": "median selected development best epoch; three-seed fixed-epoch final ensemble",
    "denominator_rule": "finite_and_strictly_positive",
    "denominator_repair": "none",
    "ready_for_n20_propagation": True,
    "execution_device": str(DEVICE),
}])

CSV_MANIFEST = (
    "19_input_and_support_audit.csv",
    "19_structure_search_validation.csv",
    "19_discovered_structures.csv",
    "19_structure_vs_n16.csv",
    "19_hyperparameter_selection_validation.csv",
    "19_selected_model_specifications.csv",
    "19_neural_seed_diagnostics.csv",
    "19_structure_stability_check.csv",
    "19_event_unaware_forecasts_validation.csv",
    "19_event_unaware_forecasts_test.csv",
    "19_forecast_metrics_validation.csv",
    "19_forecast_metrics_test_model_support.csv",
    "19_forecast_metrics_test_common_support.csv",
    "19_predictive_relevance_validation.csv",
    "19_eventless_support_audit.csv",
    "19_eventless_forecasts_validation.csv",
    "19_eventless_forecasts_test.csv",
    "19_ordinary_response_ratios.csv",
    "19_event_response_weights.csv",
    "19_event_weight_summary_clean_test.csv",
    "19_event_weight_agreement.csv",
    "19_run_metadata.csv",
)
FIGURE_MANIFEST = (
    "19_figure_19_1_structure_search_R.png",
    "19_figure_19_2_structure_search_I.png",
    "19_figure_19_3_selected_lag_orders.png",
    "19_figure_19_4_test_rmse_comparison.png",
    "19_figure_19_5_R_test_forecasts.png",
    "19_figure_19_6_I_test_forecasts.png",
    "19_figure_19_7_transformer_predictive_relevance_R.png",
    "19_figure_19_8_transformer_predictive_relevance_I.png",
    "19_figure_19_9_event_weight_medians_R.png",
    "19_figure_19_10_event_weight_medians_I.png",
)
EXPORTS = {
    "19_input_and_support_audit.csv": input_and_support_audit,
    "19_structure_search_validation.csv": structure_search_validation,
    "19_discovered_structures.csv": discovered_structures,
    "19_structure_vs_n16.csv": structure_vs_n16,
    "19_hyperparameter_selection_validation.csv": hyperparameter_selection_validation,
    "19_selected_model_specifications.csv": selected_model_specifications,
    "19_neural_seed_diagnostics.csv": neural_seed_diagnostics,
    "19_structure_stability_check.csv": structure_stability_check,
    "19_event_unaware_forecasts_validation.csv": event_unaware_forecasts_validation,
    "19_event_unaware_forecasts_test.csv": event_unaware_forecasts_test,
    "19_forecast_metrics_validation.csv": forecast_metrics_validation,
    "19_forecast_metrics_test_model_support.csv": forecast_metrics_test_model_support,
    "19_forecast_metrics_test_common_support.csv": forecast_metrics_test_common_support,
    "19_predictive_relevance_validation.csv": predictive_relevance_validation,
    "19_eventless_support_audit.csv": eventless_support_audit,
    "19_eventless_forecasts_validation.csv": eventless_forecasts_validation,
    "19_eventless_forecasts_test.csv": eventless_forecasts_test,
    "19_ordinary_response_ratios.csv": ordinary_response_ratios,
    "19_event_response_weights.csv": event_response_weights,
    "19_event_weight_summary_clean_test.csv": event_weight_summary_clean_test,
    "19_event_weight_agreement.csv": event_weight_agreement,
    "19_run_metadata.csv": run_metadata,
}
assert tuple(EXPORTS) == CSV_MANIFEST
for filename, frame in EXPORTS.items():
    path = PROCESSED / filename
    frame.to_csv(path, index=False)
    assert path.is_file()

for path in list(PROCESSED.glob("19_*.csv")):
    if path.name not in CSV_MANIFEST:
        assert path.exists() and path.is_file()
        assert path.parent.resolve() == PROCESSED.resolve()
        assert path.name.startswith("19_")
        path.unlink()
for path in list(FIGURES.glob("19_figure_*.png")):
    if path.name not in FIGURE_MANIFEST:
        assert path.exists() and path.is_file()
        assert path.parent.resolve() == FIGURES.resolve()
        assert path.name.startswith("19_figure_")
        path.unlink()

stale_N19_csv_count = len([path for path in PROCESSED.glob("19_*.csv") if path.name not in CSV_MANIFEST])
stale_N19_figure_count = len([path for path in FIGURES.glob("19_figure_*.png") if path.name not in FIGURE_MANIFEST])
assert stale_N19_csv_count == 0
assert stale_N19_figure_count == 0
assert all((PROCESSED / filename).is_file() for filename in CSV_MANIFEST)
assert all((FIGURES / filename).is_file() for filename in FIGURE_MANIFEST)
assert len(event_response_weights) > 0 and len(ordinary_response_ratios) > 0
assert event_response_weights.loc[~event_response_weights["weight_defined"].eq(True), "response_ratio"].isna().all()
print(f"Authoritative manifest: {len(CSV_MANIFEST)} CSVs and {len(FIGURE_MANIFEST)} figures")
print("Stale N19 CSVs:", stale_N19_csv_count)
print("Stale N19 figures:", stale_N19_figure_count)

Authoritative manifest: 22 CSVs and 10 figures
Stale N19 CSVs: 0
Stale N19 figures: 0


## 19.10 Empirical findings and frozen N20 handoff

The authoritative tables above freeze eight independently discovered model–target structures: RF uses R|I (L=1) and I (L=2); GB uses R|I (L=2) and I (L=2); MLP uses R|I (L=1) and I|Q (L=4); Transformer uses R|I (L=1) and I|Q|R (L=5). The Transformer therefore reproduces the N16 retained/dropped edge pattern independently, while choosing a much shorter I order than N16. Common-support TEST results favor MLP for R and Transformer for I; flexible I forecasts materially improve on N17, while R improvements are smaller and model-dependent.

Clean TEST event-weight direction is unanimous across all five backgrounds for 9 of 12 target-family cases, with disagreement concentrated in R US Employment, R US CPI, and I FOMC. Non-positive flexible event backgrounds are limited to one GB-I and one MLP-I occurrence and remain undefined. The frozen N20 handoff is 19_selected_model_specifications.csv together with the new model-specific ordinary-ratio and event-response-weight tables; N20 must adopt their model labels, parent/lag fields, and validity flags in a later task. N20 is not modified here.

In [14]:
closure_checks = {
    "Step 1 non-parametric structure discovery": len(structure_search_validation) == 384 and len(discovered_structures) == 8,
    "RF automatic structure/lag selection": len(discovered_structures.loc[discovered_structures["model"].eq("RF")]) == 2,
    "GB automatic structure/lag selection": len(discovered_structures.loc[discovered_structures["model"].eq("GB")]) == 2,
    "MLP automatic structure/lag selection": len(discovered_structures.loc[discovered_structures["model"].eq("MLP")]) == 2,
    "Transformer automatic structure/lag selection": len(discovered_structures.loc[discovered_structures["model"].eq("TRANSFORMER")]) == 2,
    "N16 independence audit": selection_audit["n16_structure_used_for_selection"] is False,
    "Step 2 OOS forecasting": len(forecast_metrics_test_common_support) == 10,
    "Step 3 eventless backgrounds/event weights": len(event_response_weights) > 0,
    "TEST leakage audit": selection_audit["test_used_for_selection"] is False,
    "Corrected N18 timing inherited": target_specific_mapping["realised_model_day"].eq(target_specific_mapping["implied_model_day"]).all(),
    "Output manifest audit": all((PROCESSED / filename).is_file() for filename in CSV_MANIFEST) and all((FIGURES / filename).is_file() for filename in FIGURE_MANIFEST),
}
assert all(closure_checks.values())
print("N19 EXERCISE 4 CLOSURE")
print("----------------------------------------------")
for label, passed in closure_checks.items():
    print(f"{label}: {'PASS' if passed else 'FAIL'}")
print("Stale N19 CSVs:", stale_N19_csv_count)
print("Stale N19 figures:", stale_N19_figure_count)
print("Execution errors: 0")
print("Ready for N20 propagation: YES")

N19 EXERCISE 4 CLOSURE
----------------------------------------------
Step 1 non-parametric structure discovery: PASS
RF automatic structure/lag selection: PASS
GB automatic structure/lag selection: PASS
MLP automatic structure/lag selection: PASS
Transformer automatic structure/lag selection: PASS
N16 independence audit: PASS
Step 2 OOS forecasting: PASS
Step 3 eventless backgrounds/event weights: PASS
TEST leakage audit: PASS
Corrected N18 timing inherited: PASS
Output manifest audit: PASS
Stale N19 CSVs: 0
Stale N19 figures: 0
Execution errors: 0
Ready for N20 propagation: YES
